In [2]:
import pandas as pd
import numpy as np 
from pathlib import Path

import warnings
warnings.filterwarnings("ignore")

In [3]:
data_path = Path("../data/raw/historical_weather.csv")

print(data_path.exists())

True


In [4]:
weather = pd.read_csv(data_path)
weather['solarForecastKwh'] = weather['shortwave_radiation'] / 100 

In [5]:
def rules_engine(stage, soc, tariff, peak_tariff, solar_kwh):
    if soc < 20 and stage <= 2: return "CHARGE"
    if solar_kwh > 5 and soc < 90: return "SOLAR_PRIORITY"
    if tariff >= peak_tariff * 0.9 and soc > 40: return "DISCHARGE"
    return "HOLD"

rows = []
sample = weather.sample(5000, random_state=42)  # sample real weather rows
for _, w in sample.iterrows():
    stage = np.random.choice(range(7))
    soc = np.random.randint(5, 101)
    tariff = np.random.choice([1.2, 1.89, 2.5, 3.87])
    rows.append({
        "solarForecastKwh": w['solarForecastKwh'],
        "outageProbability": stage / 6,
        "currentTariffRate": tariff,
        "peakTariffRate": 3.87,
        "currentSoc": soc,
        "estimatedLoad": 3.5,
        "loadSheddingStage": stage,
        "action": rules_engine(stage, soc, tariff, 3.87, w['solarForecastKwh']),
    })

training_df = pd.DataFrame(rows)
training_df.to_csv("../data/processed/training_data.csv", index=False)
training_df['action'].value_counts()

action
HOLD              3163
SOLAR_PRIORITY     831
DISCHARGE          667
CHARGE             339
Name: count, dtype: int64